In [11]:
### This Script loads the latest yelp api pull into storage bucket and then loads the data into bronze layer table.

from google.cloud import bigquery
from google.cloud import storage
from google.oauth2 import service_account
from datetime import datetime
from pathlib import Path
import os
import glob
import pandas as pd
import json

# Navigate from current notebook folder to home/devuser/data/raw
project_root = Path("/home/devuser")
# Path to your JSON key
key_path = project_root / "keys" / "daily-restaurant-insights-0659005580cc.json"
# Create credentials object
credentials = service_account.Credentials.from_service_account_file(key_path)

# Define constants
bucket_name = "daily-restaurant-insights-bucket"  # Change if needed
project_id = "daily-restaurant-insights"  # 🔁 Replace with your actual GCP project ID
dataset_id = "daily_restaurant_insights"
table_name = "bronze_yelp_raw"
table_id = f"{project_id}.{dataset_id}.{table_name}"


# Find most recent file in raw data
raw_folder = project_root / "data" / "raw"
json_files = sorted(raw_folder.glob("*.json"), key=os.path.getctime, reverse=True)
# latest_file = max(json_files, key=os.path.getctime)

if not json_files:
    raise FileNotFoundError("⚠️ No JSON files found in data/raw/")

latest_file = json_files[0]

# Load JSON & Add Ingestion Timestamp ---
try:
    with open(latest_file) as f:
        data = json.load(f)
    df = pd.DataFrame(data["businesses"])
except json.JSONDecodeError:
    df = pd.read_json(latest_file, lines=True)

# Add ingestion timestamp
df["ingestion_timestamp"] = datetime.utcnow().isoformat()

# Save Modified File Locally ---
modified_filename = latest_file.stem + "_with_ingestion.json"
modified_path = raw_folder / modified_filename
df.to_json(modified_path, orient="records", lines=True)

# Upload to Google Cloud Storage
# destination_blob_path = f"raw/{os.path.basename(latest_file)}"
destination_blob_path = f"raw/{modified_filename}"
gcs_uri = f"gs://{bucket_name}/{destination_blob_path}"

storage_client = storage.Client(credentials=credentials)
bucket = storage_client.bucket(bucket_name)
blob = bucket.blob(destination_blob_path)

try:
    blob.upload_from_filename(modified_path)
    print(f"✅ Uploaded {modified_filename} to {gcs_uri}")
except Exception as e:
    print(f"❌ Upload to GCS failed: {e}")
    raise


# Load into BigQuery


bq_client = bigquery.Client(credentials=credentials, project=project_id)

try:
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        autodetect=True,  # Auto-detect schema from the JSON
        write_disposition="WRITE_APPEND"  # Append mode
        # time_partitioning=bigquery.TimePartitioning(
        #     type_=bigquery.TimePartitioningType.DAY,
        #     field="ingestion_timestamp" #Partitioned by new field
        # ),
        # schema=[bigquery.SchemaField("ingestion_timestamp", "TIMESTAMP", mode="NULLABLE")]
        # ,schema_update_options=[
        #     bigquery.SchemaUpdateOption.ALLOW_FIELD_ADDITION,
        #     bigquery.SchemaUpdateOption.ALLOW_FIELD_RELAXATION]
    )

    # job_config.autodetect = True
    # job_config.schema = [
    # ]

    load_job = bq_client.load_table_from_uri(
        gcs_uri,
        table_id,
        job_config=job_config,
        job_id_prefix = "load_yelp_"
    )


# try:
#     load_job = bq_client.load_table_from_uri(
#         gcs_uri,
#         table_id,
#         job_config=job_config
#     )
    load_job.result()
    print(f"✅ Data loaded into BigQuery table: {table_id}")
except Exception as e:
    print(f"❌ BigQuery load failed: {e}")
    raise



✅ Uploaded yelp_sample_20250620_210633_with_ingestion.json to gs://daily-restaurant-insights-bucket/raw/yelp_sample_20250620_210633_with_ingestion.json
✅ Data loaded into BigQuery table: daily-restaurant-insights.daily_restaurant_insights.bronze_yelp_raw
